# DeepLabV3+ ResNet34 research split v1

Этот notebook запускает честный воспроизводимый stage для `archaeology_5class` через clean scripts модуля `03_multiclass_segmentation_deeplab`.

Что делает notebook:

1. проверяет GPU и зависимости;
2. клонирует/обновляет репозиторий в `/kaggle/working`;
3. находит `segmentation_dataset`;
4. проверяет frozen split artifact `splits/archaeology_5class_research_split_v1`;
5. если split CSV отсутствуют, создает split один раз через `scripts/create_research_split.py`;
6. запускает `scripts/run_research_split_v1.py` для ResNet34 old recipe;
7. показывает summaries и архивирует `runs/research_split_v1`.

Важно: notebook больше не использует старый ручной список validation-регионов. Все эксперименты идут через `--split frozen` и сохраненные CSV.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import torch
    print('python:', sys.version)
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('cuda device:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', repr(exc))

## Clone or update repository

In [ ]:
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/MataNerdy/Geodata_Archaeology_CV.git')
BRANCH = os.environ.get('BRANCH', 'main')
REPO_DIR = Path('/kaggle/working/Geodata_Archaeology_CV')

if REPO_DIR.exists():
    print('Repo already exists. Pulling latest changes...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

PROJECT_DIR = REPO_DIR
SEG_DIR = PROJECT_DIR / '03_multiclass_segmentation_deeplab'
sys.path.insert(0, str(PROJECT_DIR))

print('Using project directory:', PROJECT_DIR)
print('Segmentation dir:', SEG_DIR)
subprocess.run(['git', 'status', '--short'], cwd=PROJECT_DIR, check=True)
subprocess.run(['git', 'log', '--oneline', '-3'], cwd=PROJECT_DIR, check=True)

## Install/check dependencies

In [ ]:
# Kaggle images may not include segmentation_models_pytorch by default.
try:
    import segmentation_models_pytorch as smp
    import cv2
    import shapely
    print('segmentation_models_pytorch:', smp.__version__)
except Exception as exc:
    print('Installing requirements because import failed:', repr(exc))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SEG_DIR / 'requirements.txt')], check=True)
    import segmentation_models_pytorch as smp
    print('segmentation_models_pytorch:', smp.__version__)

## Resolve dataset and run paths

In [ ]:
def valid_data_root(path: Path) -> bool:
    return (path / 'metadata.csv').exists() and (path / 'images').is_dir() and (path / 'masks').is_dir()

candidates = []
if os.environ.get('DATA_ROOT'):
    candidates.append(Path(os.environ['DATA_ROOT']))
candidates.extend([
    Path('/kaggle/input/datasets/matanerdy/kurgans-dataset/segmentation_dataset/segmentation_dataset'),
    Path('/kaggle/input/datasets/matanerdy/kurgans-dataset/segmentation_dataset'),
    Path('/kaggle/input/kurgans-dataset/segmentation_dataset/segmentation_dataset'),
    Path('/kaggle/input/kurgans-dataset/segmentation_dataset'),
])

DATA_ROOT = next((p for p in candidates if valid_data_root(p)), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find dataset root. Set DATA_ROOT to a folder with metadata.csv/images/masks.')

RUN_ROOT = Path(os.environ.get('RUN_ROOT', str(SEG_DIR / 'runs')))
RESEARCH_ROOT = RUN_ROOT / 'research_split_v1'
SPLIT_DIR = SEG_DIR / 'splits' / 'archaeology_5class_research_split_v1'

print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT:', RUN_ROOT)
print('RESEARCH_ROOT:', RESEARCH_ROOT)
print('SPLIT_DIR:', SPLIT_DIR)

## Check or create frozen split

Этот блок можно запускать безопасно: если `train_split.csv` и `val_split.csv` уже есть, split не пересчитывается. Если CSV отсутствуют, он создается один раз и сохраняется как protocol artifact.

`test_split.csv` не создается автоматически.

In [ ]:
train_split = SPLIT_DIR / 'train_split.csv'
val_split = SPLIT_DIR / 'val_split.csv'

if train_split.exists() and val_split.exists():
    print('[split] Frozen split already exists. No recomputation.')
else:
    print('[split] Frozen split missing. Creating it once...')
    subprocess.run([
        sys.executable, 'scripts/create_research_split.py',
        '--data-root', str(DATA_ROOT),
        '--out-dir', str(SPLIT_DIR),
    ], cwd=SEG_DIR, check=True)

for name in ['train_split.csv', 'val_split.csv', 'test_split.csv', 'split_config.json', 'split_stats.md']:
    path = SPLIT_DIR / name
    print(name, 'OK' if path.exists() else 'MISSING/TODO')

## Inspect split stats

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

train_df = pd.read_csv(SPLIT_DIR / 'train_split.csv')
val_df = pd.read_csv(SPLIT_DIR / 'val_split.csv')
print('train samples:', len(train_df))
print('val samples:', len(val_df))
print('train regions:', train_df['region'].nunique())
print('val regions:', val_df['region'].nunique())

print('
Val class distribution')
display(val_df['class_name'].value_counts())
print('
Val modality distribution')
display(val_df['modality'].value_counts())
print('
Val region distribution')
display(val_df['region'].value_counts())

stats_path = SPLIT_DIR / 'split_stats.md'
if stats_path.exists():
    display(Markdown(stats_path.read_text(encoding='utf-8')))

## Run research split v1 stage

Для новой полной сессии оставь `TRAINING_GROUPS='both'`.

Если all-modalities уже посчитаны и нужно продолжить только Li-series, используй безопасный resume-режим ниже:

```python
os.environ['TRAINING_GROUPS'] = 'li'
os.environ['NUM_WORKERS'] = '0'
os.environ['RUN_POSTPROCESS_SWEEP'] = '0'
os.environ['RUN_SAMPLER_ABLATION'] = '0'
```

`NUM_WORKERS=0` выбран для Kaggle intentionally: после длинной серии worker-процессы DataLoader иногда зависают при завершении. `SKIP_EXISTING=1` не перезаписывает уже завершенные runs.

Postprocessing sweep и sampler ablation лучше запускать отдельным коротким проходом после того, как обе серии скачаны и проверены.


In [ ]:
RUN_TRAINING = os.environ.get('RUN_TRAINING', '1') == '1'
RUN_POSTPROCESS_SWEEP = os.environ.get('RUN_POSTPROCESS_SWEEP', '0') == '1'
RUN_SAMPLER_ABLATION = os.environ.get('RUN_SAMPLER_ABLATION', '0') == '1'
TRAINING_GROUPS = os.environ.get('TRAINING_GROUPS', 'li')
NUM_WORKERS = os.environ.get('NUM_WORKERS', '0')
SKIP_EXISTING = os.environ.get('SKIP_EXISTING', '1') == '1'

cmd = [
    sys.executable, 'scripts/run_research_split_v1.py',
    '--data-root', str(DATA_ROOT),
    '--run-root', str(RESEARCH_ROOT),
    '--python-bin', sys.executable,
    '--train-split-csv', str(SPLIT_DIR / 'train_split.csv'),
    '--val-split-csv', str(SPLIT_DIR / 'val_split.csv'),
    '--batch-size', '16',
    '--num-workers', NUM_WORKERS,
    '--training-groups', TRAINING_GROUPS,
]
if RUN_TRAINING:
    cmd.append('--run-training')
if SKIP_EXISTING:
    cmd.append('--skip-existing')
if RUN_POSTPROCESS_SWEEP:
    cmd.append('--run-postprocess-sweep')
if RUN_SAMPLER_ABLATION:
    cmd.append('--run-sampler-ablation')

print('TRAINING_GROUPS:', TRAINING_GROUPS)
print('NUM_WORKERS:', NUM_WORKERS)
print('$', ' '.join(map(str, cmd)))
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

process = subprocess.Popen(
    cmd,
    cwd=SEG_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)


## Stage C: postprocessing sweep for the selected validation winner

Запускай эту ячейку только после того, как seed-series завершилась. Она **не обучает модели заново**.

Runner повторно читает validation summaries, выбирает checkpoint с максимальным `weighted_competition_f1` и перебирает:

- confidence threshold: `0.00, 0.10, ..., 0.50`;
- min component area: `8, 16, 32, 64, 128, 256`;
- morphology opening: `False / True`.

Итого: `72` postprocessing-конфигурации. Результаты сохраняются в `runs/research_split_v1/postprocess_sweep/`.


In [ ]:
# Upload the selected checkpoint as a Kaggle Dataset file.
# The recursive lookup also works when Kaggle inserts dataset-owner folders.
checkpoint_name = os.environ.get('POSTPROCESS_CHECKPOINT_NAME', 'resnet34_li_seed_101.pth')
checkpoint_matches = sorted(Path('/kaggle/input').rglob(checkpoint_name))
if not checkpoint_matches:
    raise FileNotFoundError(
        f'Checkpoint {checkpoint_name!r} not found under /kaggle/input. '
        'Upload it as a Kaggle Dataset or set POSTPROCESS_CHECKPOINT_NAME.'
    )
POSTPROCESS_CHECKPOINT = checkpoint_matches[0]
print('POSTPROCESS_CHECKPOINT:', POSTPROCESS_CHECKPOINT)

sweep_cmd = [
    sys.executable, '-u', 'scripts/run_research_split_v1.py',
    '--data-root', str(DATA_ROOT),
    '--run-root', str(RESEARCH_ROOT),
    '--python-bin', sys.executable,
    '--train-split-csv', str(SPLIT_DIR / 'train_split.csv'),
    '--val-split-csv', str(SPLIT_DIR / 'val_split.csv'),
    '--batch-size', '16',
    '--num-workers', '0',
    '--run-postprocess-sweep',
    '--postprocess-checkpoint', str(POSTPROCESS_CHECKPOINT),
    '--postprocess-experiment', 'resnet34_li_seed_101',
    '--postprocess-modalities', 'Li',
]

print('$', ' '.join(map(str, sweep_cmd)))
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    sweep_cmd,
    cwd=SEG_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, sweep_cmd)


## Stage C2: selected all-modalities postprocessing sweeps

Эта ячейка отдельно прогоняет Stage C для трех сильных all-modalities checkpoints: seeds `21`, `77`, `101`. Обучение не запускается.

Перед запуском загрузи checkpoints как Kaggle Dataset files:

```text
resnet34_all_seed_21.pth
resnet34_all_seed_77.pth
resnet34_all_seed_101.pth
```

Каждая модель оценивается на полном frozen validation split с модальностями `Li,Ae,SpOr`. Для каждой модели перебираются те же `72` postprocessing-конфигурации.


In [ ]:
import shutil

all_checkpoint_names = [
    'resnet34_all_seed_21.pth',
    'resnet34_all_seed_77.pth',
    'resnet34_all_seed_101.pth',
]

all_sweep_root = RESEARCH_ROOT / 'postprocess_sweep_all_selected'
all_models_dir = all_sweep_root / 'collect_models' / 'all'
all_models_dir.mkdir(parents=True, exist_ok=True)

for stale_checkpoint in all_models_dir.glob('*'):
    if stale_checkpoint.suffix.lower() in {'.pth', '.pt', '.ckpt'}:
        stale_checkpoint.unlink()

for checkpoint_name in all_checkpoint_names:
    matches = sorted(Path('/kaggle/input').rglob(checkpoint_name))
    if not matches:
        raise FileNotFoundError(
            f'Checkpoint {checkpoint_name!r} not found under /kaggle/input. '
            'Upload the three selected all-modalities checkpoints as a Kaggle Dataset.'
        )
    source = matches[0]
    destination = all_models_dir / checkpoint_name
    shutil.copy2(source, destination)
    print('Staged:', source, '->', destination)

all_sweep_cmd = [
    sys.executable, '-u', 'scripts/collected_models_postprocess_sweep.py',
    '--data-root', str(DATA_ROOT),
    '--models-root', str(all_sweep_root / 'collect_models'),
    '--eval-root', str(all_sweep_root),
    '--task', 'archaeology_5class',
    '--image-size', '256',
    '--batch-size', '16',
    '--num-workers', '0',
    '--split', 'frozen',
    '--train-split-csv', str(SPLIT_DIR / 'train_split.csv'),
    '--val-split-csv', str(SPLIT_DIR / 'val_split.csv'),
    '--object-iou-threshold', '0.3',
]

print('$', ' '.join(map(str, all_sweep_cmd)))
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    all_sweep_cmd,
    cwd=SEG_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, all_sweep_cmd)


## Stage D: sampler ablation (manual launch)

Запускай эту ячейку вручную после Stage C. Она обучает два новых runs на том же frozen split и с тем же recipe:

- `--sampler default`;
- `--sampler weighted`.

Меняется только train sampler. Архитектура, loss, optimizer, scheduler, seed `101` и модальности `Li,Ae,SpOr` остаются фиксированными.


In [ ]:
stage_d_cmd = [
    sys.executable, '-u', 'scripts/run_sampler_ablation_stage_d.py',
    '--data-root', str(DATA_ROOT),
    '--out-root', str(RESEARCH_ROOT / 'sampler_ablation'),
    '--python-bin', sys.executable,
    '--train-split-csv', str(SPLIT_DIR / 'train_split.csv'),
    '--val-split-csv', str(SPLIT_DIR / 'val_split.csv'),
    '--modalities', 'Li,Ae,SpOr',
    '--seed', '101',
    '--batch-size', '16',
    '--num-workers', '2',
]

print('$', ' '.join(map(str, stage_d_cmd)))
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    stage_d_cmd,
    cwd=SEG_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, stage_d_cmd)


## Show summaries

In [ ]:
from IPython.display import display, Markdown, Image

summary_csv = RESEARCH_ROOT / 'research_split_v1_seed_summary.csv'
summary_md = RESEARCH_ROOT / 'research_split_v1_seed_summary.md'
selection_md = RESEARCH_ROOT / 'best_model_selection.md'

if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    display(df.sort_values('best_val_weighted_f1', ascending=False))
else:
    print('Summary CSV not found yet:', summary_csv)

for path in [summary_md, selection_md]:
    if path.exists():
        display(Markdown(path.read_text(encoding='utf-8')))
    else:
        print('Missing:', path)

sweep_pngs = sorted((RESEARCH_ROOT / 'postprocess_sweep').rglob('postprocess_sweep.png'))
if sweep_pngs:
    print('Showing:', sweep_pngs[0])
    display(Image(filename=str(sweep_pngs[0])))
else:
    print('No postprocess_sweep.png found yet')

## Zip outputs

In [ ]:
zip_path = Path('/kaggle/working/deeplab_research_split_v1_runs.zip')
if RESEARCH_ROOT.exists():
    subprocess.run(['zip', '-r', str(zip_path), str(RESEARCH_ROOT), str(SPLIT_DIR)], check=True)
    print('Saved:', zip_path)
else:
    print('Nothing to zip yet:', RESEARCH_ROOT)